# Simple LangChain Application for building retrieval-augmented generation (RAG)

**Retrieval-Augmented Generation (RAG)** system built with LangChain that combines vector stores, embeddings, and language models to retrieve and process contextual information. The implementation uses Chroma as the vector database backend and HuggingFace embeddings for semantic understanding.

In [1]:
from typing import List
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda
from langchain_groq import ChatGroq
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

import os
from dotenv import load_dotenv

from typing import List
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda


from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough


In [2]:
load_dotenv()

True

In [3]:
groq_api_key = os.getenv("GROQ_API_KEY")
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

In [4]:
documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mammal-pets-doc"}
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source": "mammal-pets-doc"}
    ),
    Document(
        page_content="Goldfish are popular pets for beginners, requiring relatively simple care.",
        metadata={"source": "fish-pets-doc"}
    ),
    Document(
        page_content="Parrots are intelligent birds capable of mimicking human speech.",
        metadata={"source": "bird-pets-doc"}
    ),
    Document(
        page_content="Rabbits are social animals that need plenty of space to hop around.",
        metadata={"source": "mammal-pets-doc"}
    ),
]

In [5]:
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

In [6]:
vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embeddings
)

In [7]:
# Basic similarity search
results = vector_store.similarity_search("Goldfish")
for result in results:
    print(result.page_content)

Goldfish are popular pets for beginners, requiring relatively simple care.
Parrots are intelligent birds capable of mimicking human speech.
Dogs are great companions, known for their loyalty and friendliness.
Cats are independent pets that often enjoy their own space.


In [8]:
# Similarity search with scores
results_with_scores = vector_store.similarity_search_with_score("Goldfish", k=1)
for doc, score in results_with_scores:
    print(f"Score: {score}, Content: {doc.page_content}")

Score: 0.639975905418396, Content: Goldfish are popular pets for beginners, requiring relatively simple care.


##### The below code works fine, just ignore the indicator. It's harmless and informational. Jupyter has its own event loop, to handel async call. 

In [9]:
# Async similarity search
results_async = await vector_store.asimilarity_search_with_score("cat", k=1)
for doc, score in results_async:
    print(f"Score: {score}, Content: {doc.page_content}")

Score: 0.9351057410240173, Content: Cats are independent pets that often enjoy their own space.


In [10]:
# Method 1: Using RunnableLambda
retriever_lambda = RunnableLambda(
    vector_store.similarity_search
).bind(k=1)

results = retriever_lambda.batch(["Goldfish"])
for result_list in results:
    for doc in result_list:
        print(doc.page_content)

Goldfish are popular pets for beginners, requiring relatively simple care.


In [11]:
# Method 2: Using as_retriever() - Recommended
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1}
)

results = retriever.batch(["Goldfish"])

for result_list in results:
    for doc in result_list:
        print(doc.page_content)

Goldfish are popular pets for beginners, requiring relatively simple care.


In [12]:
# Create prompt template
prompt = ChatPromptTemplate.from_messages([
    ("human", """Answer this question using the provided context only.
    
Context: {context}

Question: {question}""")
])

In [13]:
llm = ChatGroq(
    api_key=groq_api_key,
    model="llama-3.1-8b-instant"
)

In [15]:
# Build RAG chain
rag_chain = {
    "context": retriever,
    "question": RunnablePassthrough()
} | prompt | llm

In [16]:
rag_chain.invoke("Between goldfish and a dog, which pet would be easy to keep ?")

AIMessage(content='Based on the provided context, goldfish would be the easier pet to keep between the two options. The context states that goldfish "require relatively simple care," indicating that they are a low-maintenance pet.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 115, 'total_tokens': 158, 'completion_time': 0.093576603, 'completion_tokens_details': None, 'prompt_time': 0.007132972, 'prompt_tokens_details': None, 'queue_time': 0.050516758, 'total_time': 0.100709575}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019b6d8a-4daf-7eb0-974b-183ac9033ea1-0', usage_metadata={'input_tokens': 115, 'output_tokens': 43, 'total_tokens': 158})